# 05 — CFPB Complaint Intelligence EDA
## Consumer Financial Protection Bureau Complaint Dataset
24,665 complaints with full narratives. American Express company complaints.
Goal: Identify pain points, response patterns, and escalation signals.

In [1]:
import pandas as pd, numpy as np, sqlite3
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import re
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load and Clean ───────────────────────────────────────────────
df = pd.read_csv('../../data/raw/complaints/cfpb_complaints.csv', parse_dates=['Date received', 'Date sent to company'])
df.drop(columns=['Company public response'], inplace=True)

df.columns = [c.lower().replace(' ', '_').replace('?','').replace('/','_') for c in df.columns]
df.rename(columns={
    'consumer_complaint_narrative': 'narrative',
    'company_response_to_consumer': 'company_response',
    'timely_response': 'timely'
}, inplace=True)

df['narrative_word_count'] = df['narrative'].str.split().str.len()
df['days_to_respond'] = (df['date_sent_to_company'] - df['date_received']).dt.days

print(f"Shape: {df.shape}")
print(f"Date range: {df['date_received'].min()} to {df['date_received'].max()}")
print(f"Avg narrative length: {df['narrative_word_count'].mean():.0f} words")
print(f"Timely: {df['timely'].value_counts().to_dict()}")

Shape: (24665, 17)
Date range: 2015-03-19 22:56:29+00:00 to 2026-04-13 08:03:49+00:00
Avg narrative length: 225 words
Timely: {'Yes': 24618, 'No': 47}


In [3]:
# ── SQL Analysis ───────────────────────────────────────────────
con = sqlite3.connect(':memory:')
df.to_sql('complaints', con, index=False, if_exists='replace')

# Q1: Product distribution with response quality
q1 = pd.read_sql_query("""
    SELECT product, COUNT(*) as total,
           SUM(CASE WHEN timely='No' THEN 1 ELSE 0 END) as untimely,
           SUM(CASE WHEN company_response LIKE '%monetary%' THEN 1 ELSE 0 END) as monetary_relief,
           ROUND(100.0*SUM(CASE WHEN company_response LIKE '%monetary%' THEN 1 ELSE 0 END)/COUNT(*),2) as monetary_pct
    FROM complaints GROUP BY product ORDER BY total DESC
""", con)
print("Q1: Product Distribution & Response Quality")
print(q1.to_string(index=False))
print()

# Q2: Response type distribution
q2 = pd.read_sql_query("""
    SELECT company_response, COUNT(*) as count,
           ROUND(100.0*COUNT()/24665,2) as pct,
           ROUND(AVG(narrative_word_count),0) as avg_narrative_length,
           ROUND(AVG(days_to_respond),1) as avg_days
    FROM complaints GROUP BY company_response ORDER BY count DESC
""", con)
print("Q2: Response Type Distribution")
print(q2.to_string(index=False))
print()

# Q3: Issue frequency
q3 = pd.read_sql_query("""
    SELECT issue, COUNT(*) as count
    FROM complaints GROUP BY issue ORDER BY count DESC LIMIT 20
""", con)
print("Q3: Top 10 Issues (of 20)")
print(q3.head(10).to_string(index=False))
print()

# Q4: Submission channel
q4 = pd.read_sql_query("""
    SELECT submitted_via, COUNT(*) as count,
           ROUND(100.0*COUNT()/24665,2) as pct
    FROM complaints GROUP BY submitted_via ORDER BY count DESC
""", con)
print("Q4: Submission Channel")
print(q4.to_string(index=False))
print()

# Q5: State distribution
q5 = pd.read_sql_query("""
    SELECT state, COUNT(*) as count
    FROM complaints WHERE state IS NOT NULL
    GROUP BY state ORDER BY count DESC LIMIT 15
""", con)
print("Q5: Top 15 States")
print(q5.head(5).to_string(index=False))
print()

# Q6: Monthly complaint volume trend
q6 = pd.read_sql_query("""
    SELECT strftime('%Y-%m', date_received) as month,
           COUNT(*) as complaint_count,
           SUM(CASE WHEN timely='No' THEN 1 ELSE 0 END) as untimely_count
    FROM complaints GROUP BY month ORDER BY month
""", con)
print("Q6: Monthly Trend (first 5 months)")
print(q6.head(5).to_string(index=False))
print()

# Q7: Narrative length vs response type
q7 = pd.read_sql_query("""
    SELECT company_response,
           ROUND(AVG(narrative_word_count),0) as avg_words,
           ROUND(MIN(narrative_word_count),0) as min_words,
           ROUND(MAX(narrative_word_count),0) as max_words
    FROM complaints GROUP BY company_response
""", con)
print("Q7: Narrative Length vs Response")
print(q7.to_string(index=False))
print()

# Q8: Days to respond by product
q8 = pd.read_sql_query("""
    SELECT product, ROUND(AVG(days_to_respond),2) as avg_days,
           ROUND(MAX(days_to_respond),0) as max_days
    FROM complaints WHERE days_to_respond >= 0
    GROUP BY product ORDER BY avg_days DESC LIMIT 10
""", con)
print("Q8: Days to Respond by Product")
print(q8.to_string(index=False))


Q1: Product Distribution & Response Quality
                                                                     product  total  untimely  monetary_relief  monetary_pct
                                                 Credit card or prepaid card   7735         1             2333       30.1600
                                                                 Credit card   7582        27             1890       24.9300
                                                                Prepaid card   2688         3             1555       57.8500
                         Credit reporting or other personal consumer reports   1957         4              340       17.3700
Credit reporting, credit repair services, or other personal consumer reports   1837         1              436       23.7300
                                                             Debt collection   1677         3              561       33.4500
                                                 Checking or savings account    7

In [4]:
# ── Text Analysis (Keywords & Rule-Based) ─────────────────────────────────
stop_words = set(['the','a','an','and','or','but','in','on','at','to','for',
                  'of','with','by','from','is','was','are','were','be','been',
                  'have','has','had','do','does','did','will','would','could',
                  'should','may','might','can','not','no','my','i','me','we',
                  'they','them','their','this','that','these','those','it',
                  'its','as','if','so','up','out','about','into','through',
                  'during','before','after','above','below','between'])

all_words = []
for text in df['narrative'].dropna().sample(5000, random_state=42):
    words = re.findall(r'\b[a-zA-Z]{4,}\b', str(text).lower())
    all_words.extend([w for w in words if w not in stop_words])

word_freq = Counter(all_words).most_common(30)
word_df = pd.DataFrame(word_freq, columns=['word', 'frequency'])
print("Top 30 complaint keywords (sample of 30):")
print(word_df.head(10).to_string(index=False))

def classify_complaint(text):
    text = str(text).lower()
    if any(w in text for w in ['fraud','unauthorized','stolen','theft','scam']):
        return 'Fraud'
    elif any(w in text for w in ['billing','charge','fee','overcharge','statement']):
        return 'Billing'
    elif any(w in text for w in ['declined','blocked','limit','credit']):
        return 'Card Issues'
    elif any(w in text for w in ['rewards','points','cashback','miles']):
        return 'Rewards'
    elif any(w in text for w in ['service','representative','agent','response']):
        return 'Customer Service'
    elif any(w in text for w in ['delay','late','waiting','pending']):
        return 'Service Delay'
    else:
        return 'Other'

df['complaint_category'] = df['narrative'].apply(classify_complaint)
print("\nRule-based category distribution:")
print(df['complaint_category'].value_counts())

df['escalation_proxy'] = (
    (df['timely'] == 'No') |
    (df['company_response'] == 'Closed with monetary relief') |
    (df['narrative_word_count'] > df['narrative_word_count'].quantile(0.85))
).astype(int)
print(f"\nEscalation proxy (% flagged): {df['escalation_proxy'].mean()*100:.2f}%")

Top 30 complaint keywords (sample of 30):
       word  frequency
       xxxx      48455
   american      10703
    express      10656
       card      10353
     credit       9435
    account       8733
       amex       5908
      which       3100
information       2839
    payment       2691



Rule-based category distribution:
complaint_category
Billing             8340
Fraud               6267
Card Issues         5742
Other               2235
Customer Service     976
Rewards              683
Service Delay        422
Name: count, dtype: int64

Escalation proxy (% flagged): 30.30%


In [5]:
# ── VADER Sentiment Analysis (Full Dataset) ─────────────────────────────
sia = SentimentIntensityAnalyzer()
print("Running VADER on 24,665 narratives...")
vader_scores = df['narrative'].apply(lambda x: sia.polarity_scores(str(x)))
df['vader_neg'] = vader_scores.apply(lambda x: x['neg'])
df['vader_neu'] = vader_scores.apply(lambda x: x['neu'])
df['vader_pos'] = vader_scores.apply(lambda x: x['pos'])
df['vader_compound'] = vader_scores.apply(lambda x: x['compound'])

def vader_sentiment(compound):
    if compound <= -0.5: return 'Highly Negative'
    elif compound <= -0.1: return 'Negative'
    elif compound <= 0.1: return 'Neutral'
    elif compound <= 0.5: return 'Positive'
    else: return 'Highly Positive'

df['vader_sentiment'] = df['vader_compound'].apply(vader_sentiment)
print("\nVADER Sentiment Distribution (REAL):")
print(df['vader_sentiment'].value_counts())
print(f"\nAvg compound score: {df['vader_compound'].mean():.4f}")
print(f"Most negative complaint compound: {df['vader_compound'].min():.4f}")
print(f"Most positive complaint compound: {df['vader_compound'].max():.4f}")

con_vader = sqlite3.connect(':memory:')
df.to_sql('complaints_v', con_vader, index=False, if_exists='replace')
q_vader = '''
SELECT vader_sentiment,
       COUNT(*) as count,
       ROUND(100.0*COUNT()/24665, 2) as pct,
       ROUND(AVG(narrative_word_count), 0) as avg_words,
       SUM(CASE WHEN company_response='Closed with monetary relief'
           THEN 1 ELSE 0 END) as monetary_relief_count,
       ROUND(100.0*SUM(CASE WHEN company_response='Closed with monetary relief'
           THEN 1 ELSE 0 END)/COUNT(*), 2) as monetary_pct
FROM complaints_v
GROUP BY vader_sentiment
ORDER BY
  CASE vader_sentiment
    WHEN 'Highly Negative' THEN 1
    WHEN 'Negative' THEN 2
    WHEN 'Neutral' THEN 3
    WHEN 'Positive' THEN 4
    ELSE 5
  END
'''
vader_response = pd.read_sql(q_vader, con_vader)
print("\nVADER Sentiment vs Company Response:")
print(vader_response.to_string(index=False))

q_vader2 = '''
SELECT complaint_category, 
       ROUND(AVG(vader_compound), 4) as avg_compound,
       ROUND(MIN(vader_compound), 4) as min_compound,
       COUNT(*) as count
FROM complaints_v
GROUP BY complaint_category
ORDER BY avg_compound ASC
'''
print("\nAvg VADER score by complaint category:")
print(pd.read_sql(q_vader2, con_vader).to_string(index=False))
con_vader.close()

Running VADER on 24,665 narratives...



VADER Sentiment Distribution (REAL):
vader_sentiment
Highly Positive    9549
Highly Negative    8174
Positive           2903
Negative           2566
Neutral            1473
Name: count, dtype: int64

Avg compound score: 0.0523
Most negative complaint compound: -0.9999
Most positive complaint compound: 0.9999



VADER Sentiment vs Company Response:
vader_sentiment  count     pct  avg_words  monetary_relief_count  monetary_pct
Highly Negative   8174 33.1400   267.0000                   1545       18.9000
       Negative   2566 10.4000   138.0000                    453       17.6500
        Neutral   1473  5.9700   108.0000                    251       17.0400
       Positive   2903 11.7700   125.0000                    563       19.3900
Highly Positive   9549 38.7100   262.0000                   1588       16.6300

Avg VADER score by complaint category:
complaint_category  avg_compound  min_compound  count
             Fraud       -0.2394       -0.9999   6267
     Service Delay       -0.1916       -0.9958    422
  Customer Service       -0.1176       -0.9974    976
             Other       -0.0512       -0.9961   2235
           Billing        0.1289       -0.9997   8340
       Card Issues        0.3038       -0.9983   5742
           Rewards        0.4117       -0.9893    683


In [6]:
# ── Visualizations (Part 1) ───────────────────────────────────────────
top_products = df['product'].value_counts().head(8)
other = pd.Series({'Other': df['product'].value_counts().iloc[8:].sum()})
plot1_data = pd.concat([top_products, other]).reset_index()
plot1_data.columns = ['Product', 'Count']
fig1 = px.pie(plot1_data, names='Product', values='Count', hole=0.4, title='Complaint Volume by Product')
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.update_layout(template='plotly_white', height=450)
fig1.show()

resp = df['company_response'].value_counts().reset_index()
fig2 = px.pie(resp, names='company_response', values='count', title='Company Response to Consumer Complaints')
fig2.update_layout(template='plotly_white', height=400)
fig2.show()

fig3 = px.line(q6, x='month', y='complaint_count', title='Monthly Complaint Volume Trend', markers=True)
fig3.update_layout(template='plotly_white', height=400, xaxis_title='Month', yaxis_title='Complaints')
fig3.show()

In [7]:
# ── Visualizations (Part 2) ───────────────────────────────────────────
q3_rev = q3.sort_values('count', ascending=True)
fig4 = px.bar(q3_rev, y='issue', x='count', orientation='h', title='Top 20 Complaint Issues')
fig4.update_layout(template='plotly_white', height=600)
fig4.show()

fig5 = px.histogram(df, x='narrative_word_count', nbins=50, title='Complaint Narrative Length Distribution')
fig5.add_vline(x=df['narrative_word_count'].quantile(0.85), line_dash="dash", line_color="red", annotation_text="85th %ile")
fig5.update_layout(template='plotly_white', height=400, xaxis_title='Word Count')
fig5.show()

cat_counts = df['complaint_category'].value_counts().reset_index()
fig6 = px.pie(cat_counts, names='complaint_category', values='count', hole=0.4, title='Complaint Categories (Rule-Based)')
fig6.update_layout(template='plotly_white', height=400)
fig6.show()

In [8]:
# ── Visualizations (Part 3) ───────────────────────────────────────────
state_counts = df['state'].value_counts().reset_index()
fig7 = px.choropleth(state_counts, locations='state', locationmode='USA-states', color='count', 
                     scope="usa", title='Complaint Geographic Distribution', color_continuous_scale='Reds')
fig7.update_layout(template='plotly_white', height=500)
fig7.show()

q8_rev = q8.sort_values('avg_days', ascending=True)
fig8 = px.bar(q8_rev, y='product', x='avg_days', orientation='h', title='Average Days to Respond by Product')
fig8.update_layout(template='plotly_white', height=400)
fig8.show()

In [9]:
# ── Visualizations (VADER Sentiment) ─────────────────────────────────
fig9 = px.histogram(df, x='vader_compound', color='vader_sentiment', 
                    title='VADER Sentiment Score Distribution (All 24,665 Complaints)', nbins=50)
fig9.add_vline(x=-0.1, line_dash="dash", line_color="gray", annotation_text="Neutral")
fig9.add_vline(x=-0.5, line_dash="dash", line_color="red", annotation_text="Highly Neg")
fig9.update_layout(template='plotly_white', height=400, xaxis_title='Compound Score')
fig9.show()

fig10 = px.bar(vader_response, x='vader_sentiment', y='monetary_pct',
               title='Sentiment Level vs Monetary Relief Rate (%)', color='vader_sentiment')
fig10.update_layout(template='plotly_white', height=400, yaxis_title='% with Monetary Relief')
fig10.show()

In [10]:
# ── Save ─────────────────────────────────────────────────────
df.to_csv('../../data/processed/complaints_clean.csv', index=False)
print(f"Saved complaints_clean.csv — shape {df.shape}")
print(f"Columns: {list(df.columns)}")
con.close()

Saved complaints_clean.csv — shape (24665, 24)
Columns: ['date_received', 'product', 'sub-product', 'issue', 'sub-issue', 'narrative', 'company', 'state', 'zip_code', 'tags', 'submitted_via', 'date_sent_to_company', 'company_response', 'timely', 'complaint_id', 'narrative_word_count', 'days_to_respond', 'complaint_category', 'escalation_proxy', 'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound', 'vader_sentiment']
